# 🎙️ ROTBOTS — The Voice

---

Narration for **all** scenes (AI + found footage). The voice runs over everything.

> **🤔** What does a synthetic voice do to trust?

---

In [ ]:
# SETUP
!pip install -q edge-tts
import json, subprocess, time as _time, edge_tts
from pathlib import Path
from IPython.display import display, Markdown, Audio, HTML
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
def progress_bar(c,t,l='',w=30):
    p=c/t if t>0 else 0;f=int(w*p);return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{"█"*f}{"░"*(w-f)} {c}/{t} {l} ({p:.0%})</div>'
def progress_html(title,c,t,l='',d='',color='#4ecca3'):
    return f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>{progress_bar(c,t,l)}'+(f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{d}</div>' if d else '')+'</div>'
print('✅ Setup')

In [ ]:
# LOAD SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'video_plan.json').exists()])
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
with open(SESSION_DIR/'video_plan.json') as f: plan = json.load(f)
print(f'✅ Session: {SESSION_NAME}')

In [ ]:
# LOAD STORYBOARD
AUDIO_DIR = SESSION_DIR / 'audio'; AUDIO_DIR.mkdir(exist_ok=True)
prompts_file = SESSION_DIR / 'prompts.json'
narrations = []
if (SESSION_DIR/'storyboard.json').exists():
    with open(SESSION_DIR/'storyboard.json') as f: sb = json.load(f)
    for sc in sb:
        t=sc.get('narration','')
        if t: narrations.append({'scene':sc['scene'],'narration':t,'scene_type':sc.get('scene_type','')})
    print(f'📖 {len(narrations)} narrations for ALL scenes')
    for n in narrations:
        icon='🤖' if n['scene_type']=='ai_generated' else '🏛️' if n['scene_type']=='archive' else '📂'
        print(f'   {icon} Scene {n["scene"]}: {len(n["narration"].split())}w')
else: print('❌ No storyboard!')

In [ ]:
# VOICE
VOICES={'female_us':'en-US-AriaNeural','male_us':'en-US-GuyNeural','female_uk':'en-GB-SoniaNeural','male_uk':'en-GB-RyanNeural','documentary':'en-US-GuyNeural','dramatic':'en-GB-RyanNeural','energetic':'en-US-JennyNeural'}
CHOSEN_VOICE='documentary'
voice_name=VOICES[CHOSEN_VOICE]
print(f'🎙️ {CHOSEN_VOICE} ({voice_name})')

In [ ]:
# GENERATE
async def gen_tts(text,path,voice,rate='+0%'):
    await edge_tts.Communicate(text,voice,rate=rate).save(str(path))
def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0

prog=display(HTML(''),display_id=True); audio_files=[]; t0=_time.time()
for idx,n in enumerate(narrations):
    out=AUDIO_DIR/f'narration_{n["scene"]:03d}.mp3'
    el=_time.time()-t0;sp=idx/el if el>0 else 0;eta=(len(narrations)-idx)/sp if sp>0 else 0
    prog.update(HTML(progress_html(f'🎙️ Scene {n["scene"]}',idx,len(narrations),'narrations',f'⏱️ {el:.0f}s · ~{eta:.0f}s left')))
    try:
        await gen_tts(n['narration'],out,voice_name)
        audio_files.append({'scene':n['scene'],'path':str(out),'duration':get_dur(out),'text':n['narration']})
    except Exception as e: print(f'   ❌ {e}')

with open(SESSION_DIR/'audio_manifest.json','w') as f: json.dump({'voice':voice_name,'files':audio_files},f,indent=2)
td=sum(a['duration'] for a in audio_files)
if prompts_file.exists():
    with open(prompts_file) as f: prompts=json.load(f)
    dm={a['scene']:a['duration'] for a in audio_files}
    for p in prompts:
        if p['scene'] in dm: p['duration']=max(3,min(10,int(dm[p['scene']])+1))
    with open(prompts_file,'w') as f: json.dump(prompts,f,indent=2)
prog.update(HTML(progress_html(f'✅ {len(audio_files)} narrations, {td:.1f}s ({_time.time()-t0:.1f}s)',len(narrations),len(narrations),'narrations')))

In [ ]:
# PREVIEW
for a in audio_files:
    display(Markdown(f'### Scene {a["scene"]} ({a["duration"]:.1f}s)\n> {a["text"]}'))
    if Path(a['path']).exists(): display(Audio(a['path'],autoplay=False))
    display(Markdown('---'))

---
*ROTBOTS Workshop — LI-MA 2026*